# Run a trained policy in the MuJoCo simulator

This notebook runs a trained **ACT** policy on a simulated SO-101 arm with **Physical AI Runtime**. It shows the same steps as for a real robot, and it ends in a live dashboard where you
start the policy with one click and watch it work.

The arm sorts items in a **conveyor cell**: items ride a belt past it, and it has to drop each one into the right bin. **Red, blue and green items go into the bin of their colour, purple or cracked items into the reject bin.**

| Step | Physical AI Runtime |
|---|---|
| 1 | Start the simulation: `physicalai-mujoco start` runs as its own process and serves the robot, its cameras and a 3D viewer |
| 2 | Load the policy: `InferenceModel.from_pretrained` |
| 3 | Connect: `RobotRuntime` with the robot, the cameras and the policy as action source |
| 4 | Run: `runtime.run()` / `runtime.stop()`, from a dashboard |

The policy was trained with [Physical AI Studio](https://github.com/open-edge-platform/physical-ai-studio) on about 100 episodes recorded from the simulation's built-in demonstrator, then exported to OpenVINO.

## Setup

Until the packages are published, run this notebook from a checkout of the repository, with its own environment:

```bash
uv sync --all-packages --extra notebooks
uv run jupyter lab packages/physicalai-mujoco-plugin/examples
```

(Once `physicalai` and the plugin are on PyPI, `pip install "physicalai[plugin-mujoco,transport,notebooks]"` is all you need.)

In [ ]:
import json
import os
import subprocess
import sys
import tempfile

# ACT policy, OpenVINO export. A Hub id, or a local folder with `manifest.json` via ACT_MODEL_DIR.
MODEL_REPO = os.environ.get("ACT_MODEL_DIR") or "iMaxxgx/act-so101-conveyor-sort"

## 1. Start the simulation

`physicalai-mujoco start` loads the scene and serves the robot to other processes by name (`--name`), its cameras and the 3D viewer. `--status-json` makes it report progress as JSON lines, ending with a `ready` event that
lists the ports it picked (`--http-port 0` and `--viser-port 0` mean "any free port"). `--exit-with-parent` stops it when this notebook's pipe to it closes, so it cannot be left running by accident.
`--viewer-theme compact` minimizes the viewer's own control panel, because the dashboard below has the controls.

In [ ]:
log = tempfile.NamedTemporaryFile("w+", prefix="mujoco-sim-", suffix=".log", delete=False)  # the simulation's own log
sim = subprocess.Popen(
    [
        sys.executable, "-m", "physicalai_mujoco_plugin", "start",
        "--profile", "so101", "--scene", "conveyor_sort", "--name", "notebook-sim",
        "--status-json", "--exit-with-parent", "--http-port", "0", "--viser-port", "0",
        "--viewer-theme", "compact",
    ],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=log, text=True,
)
for line in sim.stdout:  # one JSON event per line, until `ready` or `error`
    ready = json.loads(line)
    if ready["event"] in ("ready", "error"):
        break
if ready["event"] != "ready":
    raise RuntimeError(f"{ready.get('message')} (log: {log.name})")
ready

## 2. Load the policy

`InferenceModel.from_pretrained` downloads the policy package from the Hugging Face Hub (an OpenVINO model plus a `manifest.json` that says which inputs it needs and how to preprocess them) and compiles it.

In [ ]:
from physicalai.inference import InferenceModel

policy = InferenceModel.from_pretrained(MODEL_REPO, backend="openvino", device="CPU")

## 3. Connect the robot, the cameras and the policy

A `RobotRuntime` ties them together, as on hardware:

- the **robot**: `SharedRobot` attaches to the simulation by its name,
- the **cameras**: the simulation's MJPEG streams, served like a real robot's IP cameras. Their names (`overview`, `gripper`) must match the policy's inputs,
- the **action source**: `PolicySource` runs the policy. ACT predicts a chunk of 100 actions, and `request_threshold=0.98` asks for a fresh chunk after about every second action, which this policy needs (it does not recover from a missed grasp by itself),
- the **loop rate**: 30 Hz.

In [ ]:
from physicalai.capture import IPCamera
from physicalai.robot import SharedRobot
from physicalai.runtime import ChunkedActionQueue, LerpSmoother, PolicySource, RobotRuntime, SyncExecution

http = ready["http_url"]
runtime = RobotRuntime(
    robot=SharedRobot(name=ready["name"]),
    cameras={
        "overview": IPCamera(url=f"{http}/cameras/overview/mjpeg", width=640, height=480, fps=30),
        "gripper": IPCamera(url=f"{http}/cameras/wrist/mjpeg", width=640, height=480, fps=30),
    },
    action_source=PolicySource(
        model=policy,
        execution=SyncExecution(request_threshold=0.98),
        action_queue=ChunkedActionQueue(smoother=LerpSmoother()),
    ),
    fps=30.0,
)
runtime.connect()  # attaches to the simulation and the cameras

## 4. Run it

The dashboard shows the simulation's 3D viewer (drag to orbit, scroll to zoom), the two cameras the policy sees and the score. The score counts items that ended up in the bin they belong in (**correct**), in a wrong bin (**wrong**), or off the belt (**missed**).

- **Run policy** runs `runtime.run()` and **Stop** calls `runtime.stop()`. The belt moves only while the policy runs, so it never starts with a full belt.
- **Reset** clears the belt and starts a new episode. **Belt cm/s** sets how fast items arrive: the policy was trained at 3 cm/s, so expect more misses when you speed it up.
- **Shut down** stops the policy and the simulation.

The next cell builds the dashboard with `ipywidgets`. It is folded because it is plain user-interface code: it only talks to the simulation's HTTP API and shows its MJPEG streams. Unfold it to read it.

> Running Jupyter on another machine? The viewer and camera URLs point at *that* machine's `localhost`. Forward the ports, or start the simulation with `--viser-host 0.0.0.0 --http-host 0.0.0.0`.

In [ ]:
import threading
import time
import urllib.request

import ipywidgets as widgets


def sim_api(path, body=None):
    """GET, or POST with a JSON body, on the simulation's HTTP API; returns the response body."""
    data = None if body is None else json.dumps(body).encode()
    request = urllib.request.Request(f"{http}{path}", data=data, headers={"content-type": "application/json"})
    with urllib.request.urlopen(request, timeout=5) as response:
        return response.read()


def hold_belt(hold):
    sim_api("/episode/auto-reset", {"enabled": not hold})  # pauses / resumes the belt and the item feed


def show_dashboard(run, stop, shutdown):
    """Show the viewer, the cameras, the score and the controls; `run` is called in a thread when Run is clicked."""
    state = {"thread": None, "started": 0.0, "error": None, "closed": False}
    running = lambda: state["thread"] is not None and state["thread"].is_alive()

    stream = lambda camera: widgets.HTML(f'<img src="{http}/cameras/{camera}/mjpeg" width="360">')  # the browser plays it
    viewer = widgets.HTML(f'<iframe src="{ready["viewer_url"]}" style="width:100%;height:600px;border:1px solid #ccc"></iframe>')
    side = widgets.VBox([widgets.HTML("<b>overview</b>"), stream("overview"), widgets.HTML("<b>gripper</b>"), stream("wrist")])
    status = widgets.HTML()
    run_button = widgets.Button(description="Run policy", icon="play", button_style="success")
    stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger", disabled=True)
    reset_button = widgets.Button(description="Reset", icon="refresh", tooltip="Clear the belt and start a new episode")
    shutdown_button = widgets.Button(description="Shut down", icon="power-off")
    speed = widgets.FloatSlider(value=3.0, min=0, max=10, step=0.5, description="Belt cm/s", continuous_update=False)

    def work():
        try:
            run()
        except Exception as exc:  # show any failure in the dashboard
            state["error"] = repr(exc)

    def on_run(_):
        state.update(error=None, started=time.monotonic(), thread=threading.Thread(target=work, daemon=True))
        run_button.disabled, stop_button.disabled = True, False
        state["thread"].start()
        hold_belt(False)  # the belt starts together with the policy, not before

    def on_stop(_):
        stop_button.disabled = True
        stop()
        hold_belt(True)

    def on_reset(_):
        hold_belt(True)
        sim_api("/reset", {})

    def on_shutdown(_):
        state["closed"] = True
        shutdown()
        for control in (run_button, stop_button, reset_button, shutdown_button, speed):
            control.disabled = True
        status.value = "<b>simulation stopped</b>"

    run_button.on_click(on_run)
    stop_button.on_click(on_stop)
    reset_button.on_click(on_reset)
    shutdown_button.on_click(on_shutdown)
    speed.observe(lambda change: sim_api("/conveyor/belt-speed", {"speed": change["new"] / 100}), names="value")

    def refresh():  # twice a second: the score and the button states
        while not state["closed"]:
            try:
                episode = json.loads(sim_api("/health"))["episode"]
                score = episode["score"]
                text = f"running {time.monotonic() - state['started']:.0f} s" if running() else "idle"
                if state["error"]:
                    text = f"<span style='color:#b00'>stopped: {state['error']}</span>"
                status.value = (
                    f"<b>{text}</b> &nbsp;|&nbsp; items fed {episode['spawned']} of {episode['items_per_episode']}"
                    f" &nbsp;|&nbsp; <span style='color:#080'>correct {score['correct']}</span>"
                    f" &nbsp; <span style='color:#b00'>wrong {score['wrong']}</span>"
                    f" &nbsp; <span style='color:#a60'>missed {score['missed']}</span>"
                    f" &nbsp;|&nbsp; episodes finished {episode['episode_count']}"
                )
                if not state["closed"] and not running() and stop_button.disabled is False:
                    run_button.disabled, stop_button.disabled = False, True
            except OSError:
                pass  # the simulation is not answering (yet, or any more)
            time.sleep(0.5)

    threading.Thread(target=refresh, daemon=True).start()
    panels = widgets.GridBox(
        [viewer, side], layout=widgets.Layout(grid_template_columns="minmax(480px, 1fr) 380px", grid_gap="12px", width="100%")
    )
    controls = widgets.HBox([run_button, stop_button, reset_button, speed, shutdown_button])
    return widgets.VBox([controls, status, panels])

In [ ]:
def shutdown():
    runtime.stop()
    runtime.disconnect()
    sim.stdin.close()  # closing the pipe stops the simulation (--exit-with-parent)
    sim.wait(timeout=30)


hold_belt(True)  # the belt starts moving as soon as the simulation starts: clear it and hold it until Run
sim_api("/reset", {})
show_dashboard(run=runtime.run, stop=runtime.stop, shutdown=shutdown)

## Where next

- Start another robot or scene: `physicalai-mujoco profiles` lists the robots, and the [plugin README](../README.md) describes the scenes.
- Record your own demonstrations, train in Physical AI Studio and point `MODEL_REPO` at your own export.
- Run the same policy from the command line: `physicalai run --config runtime.yaml`.